# NHL Scoring First — Data Extraction

This notebook extracts and engineers the dataset needed to analyze the cliché: **'scoring first wins games.'**

For every regular season NHL game from 2010 to 2023, we identify:
- Which team scored the first goal
- What period and time it happened
- The game context at that moment (strength state, home/away, score)
- Whether the team that scored first ultimately won

The output is a clean game-level CSV where each row is one game, ready for analysis and modeling.

In [ ]:
import pandas as pd
import numpy as np

# Load the raw play-by-play data
# This file is ~513MB — it may take a minute to load
df = pd.read_csv('Play-By-Play_NHL.csv', low_memory=False)

# Filter to regular season only
# GameID positions 4-5 equal '02' for regular season, '03' for playoffs
df = df[df['GameID'].astype(str).str[4:6] == '02'].copy()

# Extract season from GameID (first 4 digits)
df['Season'] = df['GameID'].astype(str).str[:4].astype(int)

print(f'Regular season rows loaded: {len(df):,}')
print(f'Seasons: {df["Season"].min()} to {df["Season"].max()}')
print(f'Unique games: {df["GameID"].nunique():,}')

In [ ]:
# --- STEP 1: IDENTIFY ALL GOALS ---
# Goals are flagged with typeCode 505 and Goal == 1
# We sort by GameID and gameTime so the first row per game is the first goal

goals = df[df['Goal'] == 1].copy()
goals = goals.sort_values(['GameID', 'gameTime']).reset_index(drop=True)

print(f'Total goals in dataset: {len(goals):,}')
print(f'Goals per season (sample):')
print(goals.groupby('Season').size())

In [ ]:
# --- STEP 2: IDENTIFY FIRST GOAL OF EACH GAME ---
# The first goal is simply the earliest goal by gameTime within each GameID
# We keep all relevant context columns at the moment of the first goal

first_goals = goals.groupby('GameID').first().reset_index()

# Rename columns for clarity
first_goals = first_goals.rename(columns={
    'EventTeam': 'First_Goal_Team',
    'period': 'First_Goal_Period',
    'gameTime': 'First_Goal_Time',
    'StrengthState': 'First_Goal_Strength',
    'Venue': 'First_Goal_Team_Venue',
    'ScoreState': 'ScoreState_At_First_Goal'
})

print(f'Games with a first goal identified: {len(first_goals):,}')
print()
print('Period breakdown of first goals:')
print(first_goals['First_Goal_Period'].value_counts().sort_index())

In [ ]:
# --- STEP 3: IDENTIFY GAME WINNER ---
# The winner is determined by the last goal of the game
# The team that scores the last goal has a positive ScoreState (they are ahead)
# In overtime, ScoreState == 1 for the scoring team confirms the game winner

last_goals = goals.groupby('GameID').last().reset_index()

last_goals = last_goals[['GameID', 'EventTeam', 'period', 'ScoreState']].rename(columns={
    'EventTeam': 'Game_Winner',
    'period': 'Final_Period',
    'ScoreState': 'Final_Score_Margin'
})

# Flag overtime games (period 4 or 5)
last_goals['Went_To_OT'] = (last_goals['Final_Period'] >= 4).astype(int)

print('Final period distribution (3 = regulation, 4 = OT, 5 = SO):')
print(last_goals['Final_Period'].value_counts().sort_index())

In [ ]:
# --- STEP 4: MERGE AND ENGINEER FEATURES ---

game_df = first_goals.merge(last_goals, on='GameID')

# Did the team that scored first win?
game_df['First_Goal_Team_Won'] = (
    game_df['First_Goal_Team'] == game_df['Game_Winner']
).astype(int)

# Was the first goal scored at even strength, on a power play, or shorthanded?
# StrengthState examples: 5v5 (even), 5v4 (PP), 4v5 (SH)
def classify_strength(state):
    if pd.isna(state):
        return 'Unknown'
    nums = [int(x) for x in str(state).split('v')]
    if nums[0] > nums[1]:
        return 'Power Play'
    elif nums[0] < nums[1]:
        return 'Shorthanded'
    else:
        return 'Even Strength'

game_df['First_Goal_Situation'] = game_df['First_Goal_Strength'].apply(classify_strength)

# Time remaining in the period when the first goal was scored
# Period length is 1200 seconds (20 minutes)
# gameTime is cumulative seconds from start of game
period_starts = {1: 0, 2: 1200, 3: 2400, 4: 3600, 5: 4800}
game_df['Time_Into_Period'] = game_df.apply(
    lambda row: row['First_Goal_Time'] - period_starts.get(row['First_Goal_Period'], 0),
    axis=1
)
game_df['Time_Remaining_In_Period'] = 1200 - game_df['Time_Into_Period']

# Bucket the first goal into timing windows within the period
def time_bucket(seconds_into_period):
    if seconds_into_period <= 300:
        return 'Early (0-5 min)'
    elif seconds_into_period <= 900:
        return 'Middle (5-15 min)'
    else:
        return 'Late (15-20 min)'

game_df['First_Goal_Timing'] = game_df['Time_Into_Period'].apply(time_bucket)

# Was the first goal scored by the home team?
game_df['Home_Team_Scored_First'] = (
    game_df['First_Goal_Team_Venue'] == 'Home'
).astype(int)

# Period label for readability
game_df['First_Goal_Period_Label'] = game_df['First_Goal_Period'].map({
    1: 'Period 1', 2: 'Period 2', 3: 'Period 3',
    4: 'Overtime', 5: 'Shootout'
})

print(f'Final dataset: {len(game_df):,} games')
print()
print('Overall win rate for team that scores first:')
print(f"{game_df['First_Goal_Team_Won'].mean():.1%}")
print()
print('Win rate by period of first goal:')
print(game_df.groupby('First_Goal_Period_Label')['First_Goal_Team_Won'].agg(['mean','count']).round(3))

In [ ]:
# --- STEP 5: EXPORT ---

final_cols = [
    'GameID', 'Season',
    'First_Goal_Team', 'First_Goal_Team_Venue', 'Home_Team_Scored_First',
    'First_Goal_Period', 'First_Goal_Period_Label',
    'First_Goal_Time', 'Time_Into_Period', 'Time_Remaining_In_Period',
    'First_Goal_Timing', 'First_Goal_Situation',
    'First_Goal_Strength', 'ScoreState_At_First_Goal',
    'Game_Winner', 'Final_Period', 'Final_Score_Margin',
    'Went_To_OT', 'First_Goal_Team_Won'
]

game_df[final_cols].to_csv('NHL_Scoring_First_Data.csv', index=False)

print('Export complete: NHL_Scoring_First_Data.csv')
print()
print('Dataset summary:')
print(game_df[final_cols].describe())